In [ ]:
# =====================================================================
# PHYSICAL AI DEMONSTRATION
#
# YOLO + REAL BOTTLE DETECTION + GEMINI + INTERACTIVE TARGET
# + ROBOT ANIMATION
#
# Google Colab
#
# Pipeline:
#
# Room Image
#    ↓
# YOLO detects actual bottle
#    ↓
# User selects delivery target
#    ↓
# Gemini creates high-level task plan
#    ↓
# Robot starts lower-left
#    ↓
# Robot moves to detected bottle
#    ↓
# Robot picks bottle
#    ↓
# Robot moves to selected target
#    ↓
# Robot releases bottle
#
# =====================================================================


# =====================================================================
# STEP 0
# INSTALL REQUIRED LIBRARIES
# =====================================================================

!pip -q install ultralytics google-genai pillow opencv-python-headless



In [ ]:

# =====================================================================
# STEP 1
# IMPORT LIBRARIES
# =====================================================================

import cv2
import base64
import numpy as np

from PIL import Image as PILImage
from PIL import ImageDraw

from IPython.display import display
from IPython.display import Image

from google.colab import files
from google.colab.output import eval_js
from google.colab import userdata

from google import genai

from ultralytics import YOLO

In [ ]:
# =====================================================================
# STEP 2
# GEMINI SETUP
#
# In Colab:
#
# Left sidebar
#      ↓
# Secrets
#      ↓
# Add:
#
# GEMINI_API_KEY
#
# =====================================================================

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [ ]:
# ============================================================
# UPLOAD IMAGE + OBJECT DETECTION + BOUNDING BOXES + LABELS
# ============================================================

!pip install -q ultralytics

from google.colab import files
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Upload room image
# ------------------------------------------------------------

uploaded = files.upload()

image_file = list(uploaded.keys())[0]

# Load image
image = cv2.imread(image_file)

if image is None:
    raise Exception("Could not read the uploaded image.")

# ------------------------------------------------------------
# 2. Load YOLO
# ------------------------------------------------------------

model = YOLO("yolo11n.pt")

# ------------------------------------------------------------
# 3. Detect objects
# ------------------------------------------------------------

results = model(image, conf=0.20, verbose=False)

annotated_image = image.copy()

# ------------------------------------------------------------
# 4. Draw bounding boxes and labels
# ------------------------------------------------------------

for box in results[0].boxes:

    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

    class_id = int(box.cls[0])

    object_name = model.names[class_id]

    confidence = float(box.conf[0])

    label = f"{object_name} {confidence:.2f}"

    # Bounding box
    cv2.rectangle(
        annotated_image,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        3
    )

    # Label
    cv2.putText(
        annotated_image,
        label,
        (x1, max(y1 - 10, 25)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    print(
        f"{object_name:15s} "
        f"confidence={confidence:.2f} "
        f"box=({x1},{y1},{x2},{y2})"
    )

# ------------------------------------------------------------
# 5. Display annotated image
# ------------------------------------------------------------

plt.figure(figsize=(14, 9))

plt.imshow(
    cv2.cvtColor(
        annotated_image,
        cv2.COLOR_BGR2RGB
    )
)

plt.title("Robot Vision - Object Detection")
plt.axis("off")
plt.show()